## 1. Setup environment & imports

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

try:
    import faiss
except Exception:
    faiss = None

try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None

try:
    import tensorflow as tf
except Exception:
    tf = None

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

BASE_DIR = Path('/mnt/e/12 - Magang/Dicoding/Capstone Project')
DATA_DIR = BASE_DIR / 'Data'
ARTIFACT_DIR = BASE_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train_ready_dataset.csv'
USER_SCHEMA_PATH = DATA_DIR / 'user_profile_features_schema.csv'

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 120)
np.random.seed(42)

ALERGEN_COLUMNS = [
    'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_peanut',
    'contains_seafood', 'contains_egg', 'contains_soy', 'contains_celery',
    'contains_mustard', 'contains_sesame', 'contains_sulfite', 'contains_other',
    'contains_unknown'
]
ALLERGY_VOCAB = ['gluten', 'dairy', 'nuts', 'peanut', 'seafood', 'egg', 'soy', 'celery', 'mustard', 'sesame', 'sulfite', 'other', 'unknown']


def normalize_text(value):
    if pd.isna(value):
        return ''
    value = unicodedata.normalize('NFKC', str(value)).lower()
    value = re.sub(r'[^\w\s-]', ' ', value)
    value = re.sub(r'\s+', ' ', value).strip()
    return value


def normalize_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {'1', 'true', 'yes', 'y', 't'}


def parse_vector(value):
    if pd.isna(value):
        return []
    return [item.strip().lower() for item in str(value).split(',') if item.strip()]


def compute_macro_calories(frame):
    return 4 * frame['protein_100g'] + 9 * frame['fat_100g'] + 4 * frame['carbohydrate_100g']


def food_matches_allergy(user_allergies, food_row):
    for allergen in user_allergies:
        if allergen in food_row.index and bool(food_row[allergen]):
            return True
        if f'allergen_kw_{allergen}' in food_row.index and bool(food_row[f'allergen_kw_{allergen}']):
            return True
    return False

print('Setup ready:', ARTIFACT_DIR)

## 2. Load datasets

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
user_schema_df = pd.read_csv(USER_SCHEMA_PATH)

print('train_df shape:', train_df.shape)
print('user_schema_df shape:', user_schema_df.shape)
display(train_df.head(3))
display(user_schema_df.head(3))

## 3. Quick EDA and data cleaning

In [ ]:
train = train_df.copy()
users = user_schema_df.copy()

for column in ['calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'calorie_macro_diff', 'confidence', 'allergen_match_count']:
    if column in train.columns:
        train[column] = pd.to_numeric(train[column], errors='coerce')

for column in ['age', 'height_cm', 'weight_kg', 'bmr', 'tdee', 'target_calorie', 'protein_target_g', 'fat_target_g', 'carb_target_g', 'total_macro_kcal', 'macro_cal_diff']:
    if column in users.columns:
        users[column] = pd.to_numeric(users[column], errors='coerce')

for column in ['zero_calorie_flag', 'calorie_inconsistent_flag', 'carb_floored_flag']:
    if column in train.columns:
        train[column] = train[column].map(normalize_bool)
    if column in users.columns:
        users[column] = users[column].map(normalize_bool)

for column in ALERGEN_COLUMNS:
    if column in train.columns:
        train[column] = train[column].map(normalize_bool)

train['image_url_valid'] = train['image_url'].fillna('').map(lambda value: str(value).startswith('http'))
train['food_name_clean'] = train['food_name_clean'].fillna(train['food_name']).map(normalize_text)
train['food_name_std'] = train['food_name_clean']
users['allergy_vector_list'] = users['allergy_vector'].map(parse_vector)

train['macro_calories_calc'] = compute_macro_calories(train)
train['macro_calorie_delta'] = train['calories_100g'] - train['macro_calories_calc']
train['large_macro_delta_flag'] = train['macro_calorie_delta'].abs() > 25
train['corrected_calories'] = np.where(train['large_macro_delta_flag'], train['macro_calories_calc'], train['calories_100g'])

print('Large macro delta rows:', int(train['large_macro_delta_flag'].sum()))
display(train[['food_name', 'calories_100g', 'macro_calories_calc', 'corrected_calories']].head(5))
display(users[['age', 'gender', 'goal', 'target_calorie', 'allergy_vector']].head(5))

## 4. Standardize, allergen mapping, embeddings, and features

In [ ]:
train['food_text'] = (train['food_name_std'].fillna('') + ' ' + train['source'].fillna('').map(normalize_text)).str.strip()

def expand_abbrev(text):
    replacements = {'kw': 'kualitas', 'sdg': 'sedang', 'dgn': 'dengan'}
    tokens = [replacements.get(token, token) for token in normalize_text(text).split()]
    return ' '.join(tokens)

train['food_text'] = train['food_text'].map(expand_abbrev)
train['food_tokens'] = train['food_text'].str.split()

allergen_keywords = {
    'gluten': ['gandum', 'roti', 'mie', 'biskuit', 'tepung'],
    'dairy': ['susu', 'keju', 'mentega', 'krim'],
    'nuts': ['kacang', 'mede', 'kenari', 'almond'],
    'peanut': ['kacang tanah', 'selai kacang'],
    'seafood': ['ikan', 'udang', 'cumi', 'kepiting', 'kerang'],
    'egg': ['telur'],
    'soy': ['tahu', 'tempe', 'kedelai'],
    'celery': ['seledri'],
    'mustard': ['mustard'],
    'sesame': ['wijen'],
    'sulfite': ['sulfit', 'sulfite'],
    'other': ['alkohol', 'gelatin'],
    'unknown': []
}

train['allergen_mask'] = train[ALERGEN_COLUMNS].fillna(False).astype(bool).any(axis=1)
for allergen, keywords in allergen_keywords.items():
    column_name = f'allergen_kw_{allergen}'
    train[column_name] = False if not keywords else train['food_name_std'].apply(lambda value: any(keyword in value for keyword in keywords))

train['allergen_kw_mask'] = train[[f'allergen_kw_{allergen}' for allergen in allergen_keywords]].any(axis=1)
train['allergen_safe_flag'] = ~(train['allergen_mask'] | train['allergen_kw_mask'])

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(train['food_text'].fillna(''))

sentence_model = None
sentence_embeddings = None
if SentenceTransformer is not None:
    try:
        sentence_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        sentence_embeddings = sentence_model.encode(train['food_text'].tolist(), normalize_embeddings=True)
    except Exception:
        sentence_embeddings = None

numeric_cols = ['corrected_calories', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'macro_calories_calc', 'macro_calorie_delta']
feature_matrix = train[numeric_cols].fillna(0.0)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_matrix)

if faiss is not None:
    dense_text = tfidf_matrix.toarray().astype('float32')
    faiss.normalize_L2(dense_text)
    candidate_index = faiss.IndexFlatIP(dense_text.shape[1])
    candidate_index.add(dense_text)
    candidate_index_kind = 'faiss_ip'
else:
    candidate_index = NearestNeighbors(n_neighbors=20, metric='cosine')
    candidate_index.fit(tfidf_matrix)
    candidate_index_kind = 'sklearn_nn'

print('TF-IDF shape:', tfidf_matrix.shape)
print('Scaled feature shape:', scaled_features.shape)
print('Candidate index:', candidate_index_kind)
display(train[['food_id', 'food_name', 'allergen_safe_flag', 'corrected_calories']].head(5))

## 5. Ranking model training and evaluation

In [ ]:
sample_users = users.head(min(len(users), 30)).copy()
sample_foods = train.sample(min(len(train), 800), random_state=42).copy()
pairs = sample_users.merge(sample_foods, how='cross', suffixes=('_user', '_food'))

pairs['calorie_gap'] = (pairs['corrected_calories'] - pairs['target_calorie']).abs()
pairs['protein_gap'] = (pairs['protein_100g'] - pairs['protein_target_g']).abs()
pairs['fat_gap'] = (pairs['fat_100g'] - pairs['fat_target_g']).abs()
pairs['carb_gap'] = (pairs['carbohydrate_100g'] - pairs['carb_target_g']).abs()
pairs['allergen_penalty'] = pairs['allergen_mask'].astype(int) + (~pairs['allergen_safe_flag']).astype(int)
pairs['heuristic_label'] = 1 / (1 + 0.002 * pairs['calorie_gap'] + 0.01 * pairs['protein_gap'] + 0.005 * pairs['fat_gap'] + 0.005 * pairs['carb_gap'] + 2 * pairs['allergen_penalty'])

pair_features = pairs[['calorie_gap', 'protein_gap', 'fat_gap', 'carb_gap', 'allergen_penalty', 'macro_calorie_delta']].fillna(0.0)
X_train, X_valid, y_train, y_valid = train_test_split(pair_features, pairs['heuristic_label'], test_size=0.2, random_state=42)

if LGBMRegressor is not None:
    ranking_model = LGBMRegressor(n_estimators=250, learning_rate=0.05, num_leaves=31, random_state=42)
    ranking_model.fit(X_train, y_train)
    valid_pred = ranking_model.predict(X_valid)
else:
    from sklearn.ensemble import HistGradientBoostingRegressor
    ranking_model = HistGradientBoostingRegressor(random_state=42)
    ranking_model.fit(X_train, y_train)
    valid_pred = ranking_model.predict(X_valid)


def precision_at_k(relevance, k=10):
    relevance = np.asarray(relevance)[:k]
    return float(relevance.mean()) if len(relevance) else 0.0


def average_precision_at_k(relevance, k=10):
    relevance = np.asarray(relevance)[:k]
    if not relevance.any():
        return 0.0
    hits = 0
    scores = []
    for idx, item in enumerate(relevance, start=1):
        if item:
            hits += 1
            scores.append(hits / idx)
    return float(np.mean(scores)) if scores else 0.0


def ndcg_at_k(relevance, k=10):
    relevance = np.asarray(relevance)[:k]
    return float(relevance.sum() / max(len(relevance), 1))

print('Validation MAE:', float(np.mean(np.abs(y_valid - valid_pred))))
print('Validation precision@10 proxy:', precision_at_k((valid_pred > np.median(valid_pred)).astype(int), 10))
print('Validation MAP@10 proxy:', average_precision_at_k((valid_pred > np.median(valid_pred)).astype(int), 10))
print('Validation NDCG@10 proxy:', ndcg_at_k((valid_pred > np.median(valid_pred)).astype(int), 10))

## 6. Inference, unit tests, export, and demo

Bagian ini dipakai untuk menjalankan rekomendasi per user, mengecek test fungsi dasar, menyimpan artefak model, dan menampilkan hasil contoh.

In [ ]:
def recommend_for_user(user_profile, top_k=10):
    user_allergies = set(user_profile.get('allergy_vector_list', []))
    target_calorie = float(user_profile.get('target_calorie', user_profile.get('tdee', 2000.0)) or 2000.0)
    calorie_low = 0.75 * target_calorie
    calorie_high = 1.25 * target_calorie

    candidate_pool = train[(train['corrected_calories'] >= calorie_low / 3) & (train['corrected_calories'] <= calorie_high / 3)].copy()
    if candidate_pool.empty:
        candidate_pool = train.copy()

    candidate_pool['allergy_blocked'] = candidate_pool.apply(lambda row: food_matches_allergy(user_allergies, row), axis=1)
    candidate_pool = candidate_pool[~candidate_pool['allergy_blocked']].copy()

    candidate_pool['calorie_gap'] = (candidate_pool['corrected_calories'] - target_calorie / 3).abs()
    candidate_pool['protein_gap'] = (candidate_pool['protein_100g'] - float(user_profile.get('protein_target_g', 0.0) or 0.0) / 3).abs()
    candidate_pool['fat_gap'] = (candidate_pool['fat_100g'] - float(user_profile.get('fat_target_g', 0.0) or 0.0) / 3).abs()
    candidate_pool['carb_gap'] = (candidate_pool['carbohydrate_100g'] - float(user_profile.get('carb_target_g', 0.0) or 0.0) / 3).abs()
    candidate_pool['allergen_penalty'] = candidate_pool['allergen_mask'].astype(int) + (~candidate_pool['allergen_safe_flag']).astype(int)

    model_input = candidate_pool[['calorie_gap', 'protein_gap', 'fat_gap', 'carb_gap', 'allergen_penalty', 'macro_calorie_delta']].fillna(0.0)
    try:
        candidate_pool['model_score'] = ranking_model.predict(model_input)
    except Exception:
        candidate_pool['model_score'] = 0.0

    candidate_pool['final_score'] = 0.6 * candidate_pool['model_score'] + 0.4 * (1 / (1 + candidate_pool['calorie_gap']))
    candidate_pool['reason'] = np.where(candidate_pool['allergen_safe_flag'], 'aman dari alergen terdeteksi dan dekat dengan target kalori', 'perlu review manual karena alergen terdeteksi')

    return candidate_pool.sort_values(['final_score', 'corrected_calories'], ascending=[False, True]).head(top_k)[[
        'food_id', 'food_name', 'corrected_calories', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'allergen_safe_flag', 'final_score', 'reason'
    ]].reset_index(drop=True)


def test_data_loader():
    assert not train.empty
    assert not users.empty
    assert 'food_id' in train.columns
    assert 'target_calorie' in users.columns


def test_compute_calories_correctness():
    frame = pd.DataFrame({'protein_100g': [10.0], 'fat_100g': [5.0], 'carbohydrate_100g': [20.0]})
    assert compute_macro_calories(frame).iloc[0] == 4 * 10 + 9 * 5 + 4 * 20


def test_allergen_filter_behavior():
    sample = train[train['contains_seafood'].astype(bool)]
    if not sample.empty:
        assert food_matches_allergy({'seafood'}, sample.iloc[0]) is True


def test_recommend_for_user_deterministic():
    user = users.iloc[0].copy()
    rec1 = recommend_for_user(user, top_k=5)
    rec2 = recommend_for_user(user, top_k=5)
    assert rec1['food_id'].tolist() == rec2['food_id'].tolist()

artifact_manifest = {
    'vectorizer_path': str(ARTIFACT_DIR / 'food_tfidf_vectorizer.joblib'),
    'scaler_path': str(ARTIFACT_DIR / 'numeric_scaler.joblib'),
    'model_path': str(ARTIFACT_DIR / 'ranking_model.joblib'),
    'data_path': str(TRAIN_PATH),
    'user_schema_path': str(USER_SCHEMA_PATH),
    'candidate_index_type': candidate_index_kind
}

joblib.dump(vectorizer, ARTIFACT_DIR / 'food_tfidf_vectorizer.joblib')
joblib.dump(scaler, ARTIFACT_DIR / 'numeric_scaler.joblib')
joblib.dump(ranking_model, ARTIFACT_DIR / 'ranking_model.joblib')
with open(ARTIFACT_DIR / 'manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(artifact_manifest, handle, indent=2)

if tf is not None:
    user_numeric_input = tf.keras.Input(shape=(4,), name='user_numeric')
    food_numeric_input = tf.keras.Input(shape=(4,), name='food_numeric')
    merged = tf.keras.layers.Concatenate()([user_numeric_input, food_numeric_input])
    merged = tf.keras.layers.Dense(32, activation='relu')(merged)
    merged = tf.keras.layers.Dense(16, activation='relu')(merged)
    score_output = tf.keras.layers.Dense(1, activation='linear', name='compatibility_score')(merged)
    tf_model = tf.keras.Model([user_numeric_input, food_numeric_input], score_output)
    tf_model.save(ARTIFACT_DIR / 'nutrimatch_model.keras')
else:
    tf_model = None

print('Artifacts saved to', ARTIFACT_DIR)
print('Recommended sample output:')
display(recommend_for_user(users.iloc[0], top_k=5))

## 7. Performance and scaling

Catatan terakhir ini mengukur latency inference sederhana dan menyimpan tips untuk memperbesar dataset tanpa mengubah alur inti notebook.

In [ ]:
import time

start = time.perf_counter()
_ = recommend_for_user(users.iloc[0].copy(), top_k=10)
latency_ms = (time.perf_counter() - start) * 1000
memory_estimate_mb = (tfidf_matrix.data.nbytes + scaled_features.nbytes) / (1024 ** 2)

scaling_tips = [
    'Pakai FAISS IVF atau HNSW jika jumlah makanan bertambah besar.',
    'Simpan embedding dan scaler ke disk agar inference ringan.',
    'Pisahkan candidate retrieval dan ranking supaya latency lebih stabil.',
    'Gunakan batch inference saat banyak user masuk sekaligus.',
    'Pindahkan scoring ke API jika notebook sudah siap produksi.'
]

print(f'Approx inference latency: {latency_ms:.2f} ms')
print(f'Approx working memory footprint: {memory_estimate_mb:.2f} MB')
for tip in scaling_tips:
    print('-', tip)